# 📘 Full Fine-Tuning: Updating Every Parameter

## What is Full Fine-Tuning?

Full fine-tuning is the **most straightforward** approach to adapting a pre-trained language model:
you take the entire model and update **all parameters** on your downstream task.

### When to Use Full Fine-Tuning
| Use Case | Recommendation |
|----------|---------------|
| Abundant task-specific data (>10K examples) | ✅ Full FT excels |
| Small model (<500M params) that fits in memory | ✅ Full FT is practical |
| Need maximum task performance | ✅ Full FT has highest ceiling |
| Limited GPU memory (<8GB) with large model | ❌ Use LoRA/QLoRA instead |
| Need to preserve general knowledge | ⚠️ Risk of catastrophic forgetting |
| Few-shot scenario (<100 examples) | ❌ High overfitting risk |

### What You'll Learn in This Notebook
1. How to fully fine-tune SmolLM2-135M for instruction following
2. Memory consumption and training time profiling
3. Before/after generation quality comparison
4. 🔴 **Failure Point #1**: Catastrophic forgetting — when your model "forgets" everything else
5. 🔴 **Failure Point #2**: Overfitting & learning rate divergence — memorization vs. learning

---
**Model**: `HuggingFaceTB/SmolLM2-135M-Instruct` (135M parameters — fits on any GPU!)
**Dataset**: `databricks/dolly-15k` (instruction-following, 1000-sample subset)
**Time**: ~15–30 minutes on T4 GPU

In [1]:
# ============================================================
# 📦 Colab Setup — Run this cell first!
# ============================================================
# Uncomment the line below if running in Google Colab
# !pip install -q torch transformers datasets accelerate evaluate matplotlib pyyaml

import importlib
required = ["torch", "transformers", "datasets", "accelerate", "yaml", "matplotlib"]
missing = [pkg for pkg in required if importlib.util.find_spec(pkg) is None]
if missing:
    print(f"⚠️  Missing packages: {missing}")
    print("Run: pip install torch transformers datasets accelerate evaluate matplotlib pyyaml")
else:
    print("✅ All required packages are installed!")

✅ All required packages are installed!


In [2]:
# ============================================================
# 🔧 Imports and Device Setup
# ============================================================
import sys
import os
import time
import warnings
warnings.filterwarnings("ignore")

# Add project src to path
sys.path.insert(0, os.path.abspath("../src"))

import torch
import yaml
import matplotlib.pyplot as plt
from transformers import AutoModelForCausalLM, AutoTokenizer, Trainer, DataCollatorForLanguageModeling
from datasets import load_dataset

# Import our project utilities
from data_utils import load_and_format_dataset, apply_chat_template, compute_dataset_statistics
from training_utils import setup_device, get_device_info, get_training_args, MemoryMonitorCallback, TrainingMetricsCallback
from visualization import plot_training_curves, plot_memory_comparison

# Device setup
device = setup_device()
device_info = get_device_info()
print("=" * 50)
print("🖥️  Device Configuration")
print("=" * 50)
for k, v in device_info.items():
    print(f"  {k}: {v}")
print("=" * 50)

print(f"   Supports: MPS (Apple), CUDA (NVIDIA), cpu (fallback)")

🖥️  Device Configuration
  device: mps
  device_type: mps
  backend: Metal Performance Shaders (Apple Silicon)
  unified_memory: True
   Supports: MPS (Apple), CUDA (NVIDIA), cpu (fallback)


In [3]:
# ============================================================
# ⚙️  Load Configuration
# ============================================================
with open("../configs/full_finetune.yaml", "r") as f:
    config = yaml.safe_load(f)

# Determine if we need low-resource mode
# Set LOW_RESOURCE = True if you have ≤6GB GPU memory or are on Colab free tier
COMPACT_NOTEBOOK_RUN = True  # Keeps local validation runs practical while preserving the workflow
LOW_RESOURCE = False

# Auto-detect: if on CUDA with <8GB, use low resource
if device.type == "cuda":
    vram_gb = torch.cuda.get_device_properties(0).total_mem / 1e9
    if vram_gb < 8:
        LOW_RESOURCE = True
        print(f"⚠️  Detected {vram_gb:.1f}GB VRAM — enabling LOW_RESOURCE mode")

# Apply config
MODEL_NAME = config["model"]["name"]
if LOW_RESOURCE:
    BATCH_SIZE = config["low_resource"]["batch_size"]
    GRAD_ACCUM = config["low_resource"]["gradient_accumulation_steps"]
    MAX_SEQ_LEN = config["low_resource"]["max_seq_length"]
    GRAD_CHECKPOINT = config["low_resource"]["gradient_checkpointing"]
else:
    BATCH_SIZE = config["training"]["batch_size"]
    GRAD_ACCUM = config["training"]["gradient_accumulation_steps"]
    MAX_SEQ_LEN = config["training"]["max_seq_length"]
    GRAD_CHECKPOINT = False

NUM_EPOCHS = 1 if COMPACT_NOTEBOOK_RUN else config["training"]["num_epochs"]
LEARNING_RATE = config["training"]["learning_rate"]

print(f"📋 Configuration:")
print(f"   Model: {MODEL_NAME}")
print(f"   Epochs: {NUM_EPOCHS}")
print(f"   Batch size: {BATCH_SIZE}")
print(f"   Gradient accumulation: {GRAD_ACCUM}")
print(f"   Effective batch size: {BATCH_SIZE * GRAD_ACCUM}")
print(f"   Learning rate: {LEARNING_RATE}")
print(f"   Max seq length: {MAX_SEQ_LEN}")
print(f"   Low resource mode: {LOW_RESOURCE}")
print(f"   Compact notebook run: {COMPACT_NOTEBOOK_RUN}")
print(f"   Gradient checkpointing: {GRAD_CHECKPOINT}")

📋 Configuration:
   Model: HuggingFaceTB/SmolLM2-135M-Instruct
   Epochs: 1
   Batch size: 4
   Gradient accumulation: 4
   Effective batch size: 16
   Learning rate: 2e-05
   Max seq length: 512
   Low resource mode: False
   Compact notebook run: True
   Gradient checkpointing: False


## 🧠 How Full Fine-Tuning Works

### The Big Picture

In full fine-tuning, we take a pre-trained model and continue training on task-specific data,
updating **every single parameter** via backpropagation.

```
Pre-trained Model (135M params)  →  Task-Specific Training  →  Fine-tuned Model (135M params)
     All weights frozen: ❌              All weights updated: ✅        Same architecture
```

### Parameter Count Comparison

| Method | Trainable Parameters | Memory Needed | Training Speed |
|--------|---------------------|---------------|----------------|
| Full Fine-Tuning | 100% (~135M) | High (full model + optimizer states) | Slowest |
| LoRA (r=8) | ~0.5% (~700K) | Low (base + small adapters) | Fast |
| QLoRA | ~0.5% (4-bit base) | Very Low | Fast |
| Prompt Tuning | ~0.01% (~10K) | Minimal | Fastest |

### Why Full FT for SmolLM2-135M?

With only 135M parameters, SmolLM2-135M is small enough that full fine-tuning is
actually **practical** even on consumer hardware:
- Model weights (fp16): ~270 MB
- Optimizer states (Adam): ~540 MB  
- Gradients: ~270 MB
- **Total**: ~1.1 GB (before activations)

This makes it an ideal model for learning the fundamentals before moving to
parameter-efficient methods in later notebooks.

In [4]:
# ============================================================
# 📚 Load and Prepare Dataset
# ============================================================
print("Loading databricks/dolly-15k dataset...")
dataset_max_samples = 128 if COMPACT_NOTEBOOK_RUN else 1000
dataset = load_and_format_dataset("databricks/dolly-15k", split="train", max_samples=dataset_max_samples)
print(f"✅ Loaded {len(dataset)} examples")
print(f"   Columns: {dataset.column_names}")
print(f"   Sample categories: {list(set(dataset['category'][:20]))}")

# Show a sample
print("\n📝 Sample entry:")
print(f"   Instruction: {dataset[0]['instruction'][:100]}...")
print(f"   Response: {dataset[0]['output'][:100]}...")

'[SSL: CERTIFICATE_VERIFY_FAILED] certificate verify failed: self-signed certificate in certificate chain (_ssl.c:1010)' thrown while requesting HEAD https://huggingface.co/datasets/databricks/dolly-15k/resolve/main/README.md
Retrying in 1s [Retry 1/5].


Loading databricks/dolly-15k dataset...
✅ Loaded 128 examples
   Columns: ['category', 'instruction', 'response', 'output']
   Sample categories: ['classification', 'open_qa', 'summarization', 'brainstorming', 'closed_qa']

📝 Sample entry:
   Instruction: Explain what fine-tuning means for a language model in the context of instruction tuning....
   Response: Fine-tuning adapts a pre-trained model to instruction tuning by continuing training on task-specific...


In [5]:
# ============================================================
# 🔤 Tokenize with Chat Template
# ============================================================
print("Loading tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

# Apply chat template
system_prompt = "You are a helpful assistant. Provide clear and concise answers."
formatted_dataset = dataset.map(
    lambda examples: apply_chat_template(examples, tokenizer, system_prompt),
    batched=True,
    remove_columns=dataset.column_names,
)
print(f"✅ Formatted {len(formatted_dataset)} examples with chat template")
print(f"\n📝 Sample formatted text (first 300 chars):")
print(formatted_dataset[0]['text'][:300])

Loading tokenizer...


Map:   0%|          | 0/128 [00:00<?, ? examples/s]

✅ Formatted 128 examples with chat template

📝 Sample formatted text (first 300 chars):
<|im_start|>system
You are a helpful assistant. Provide clear and concise answers.<|im_end|>
<|im_start|>user
Explain what fine-tuning means for a language model in the context of instruction tuning.<|im_end|>
<|im_start|>assistant
Fine-tuning adapts a pre-trained model to instruction tuning by cont


In [6]:
# ============================================================
# ✂️  Tokenize and Create Train/Val Split
# ============================================================
def tokenize_function(examples):
    """Tokenize text and create labels for causal LM training."""
    tokenized = tokenizer(
        examples["text"],
        truncation=True,
        max_length=MAX_SEQ_LEN,
        padding="max_length",
    )
    # For causal LM, labels = input_ids (shifted internally by the model)
    tokenized["labels"] = tokenized["input_ids"].copy()
    return tokenized

tokenized_dataset = formatted_dataset.map(
    tokenize_function,
    batched=True,
    remove_columns=["text"],
)

# Train/validation split (90/10)
split = tokenized_dataset.train_test_split(test_size=0.1, seed=42)
train_dataset = split["train"]
eval_dataset = split["test"]

print(f"📊 Dataset split:")
print(f"   Train: {len(train_dataset)} examples")
print(f"   Eval:  {len(eval_dataset)} examples")
print(f"   Sequence length: {MAX_SEQ_LEN} tokens")

Map:   0%|          | 0/128 [00:00<?, ? examples/s]

📊 Dataset split:
   Train: 115 examples
   Eval:  13 examples
   Sequence length: 512 tokens


## 📊 Pre-Training Baseline

Before fine-tuning, let's see how the model responds to our test prompts.
We'll compare these with post-training outputs later.

In [7]:
# ============================================================
# 🧪 Generate Baseline Responses (BEFORE fine-tuning)
# ============================================================
print("Loading pre-trained model for baseline...")
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16 if device.type == "cuda" else torch.float32,
).to(device)

# Count parameters
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"\n📐 Model Parameters:")
print(f"   Total: {total_params:,} ({total_params/1e6:.1f}M)")
print(f"   Trainable: {trainable_params:,} ({trainable_params/1e6:.1f}M)")
print(f"   Trainable %: {100 * trainable_params / total_params:.1f}%")

# Test prompts for comparison
test_prompts = [
    "Explain what photosynthesis is in simple terms.",
    "Write a Python function to calculate the factorial of a number.",
    "What are the benefits of exercise for mental health?",
    "Summarize the plot of Romeo and Juliet in 2 sentences.",
    "What is the capital of France and why is it famous?",
]

def generate_responses(model, prompts, max_new_tokens=150):
    """Generate responses for a list of prompts."""
    model.eval()
    responses = []
    for prompt in prompts:
        messages = [{"role": "user", "content": prompt}]
        text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        inputs = tokenizer(text, return_tensors="pt").to(device)
        with torch.no_grad():
            outputs = model.generate(
                **inputs,
                max_new_tokens=max_new_tokens,
                do_sample=True,
                temperature=0.7,
                top_p=0.9,
                pad_token_id=tokenizer.eos_token_id,
            )
        response = tokenizer.decode(outputs[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True)
        responses.append(response.strip())
    return responses

print("\n🔍 Generating baseline responses...")
baseline_responses = generate_responses(model, test_prompts)

print("\n" + "=" * 60)
print("📋 BASELINE RESPONSES (Before Fine-Tuning)")
print("=" * 60)
for i, (prompt, response) in enumerate(zip(test_prompts, baseline_responses)):
    print(f"\n[{i+1}] Prompt: {prompt}")
    print(f"    Response: {response[:200]}")
    print("-" * 60)

# Free model to save memory before training
del model
if device.type == "cuda":
    torch.cuda.empty_cache()
elif device.type == "mps":
    torch.mps.empty_cache()

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


Loading pre-trained model for baseline...


Loading weights:   0%|          | 0/272 [00:00<?, ?it/s]


📐 Model Parameters:
   Total: 134,515,008 (134.5M)
   Trainable: 134,515,008 (134.5M)
   Trainable %: 100.0%

🔍 Generating baseline responses...

📋 BASELINE RESPONSES (Before Fine-Tuning)

[1] Prompt: Explain what photosynthesis is in simple terms.
    Response: Photosynthesis is a process used by plants, algae, and some bacteria to make their food. It's a natural process that plants, algae, and some bacteria use to convert light energy from the sun into chem
------------------------------------------------------------

[2] Prompt: Write a Python function to calculate the factorial of a number.
    Response: ```python
def factorial(n):
    if n == 0:  # base case
        return 1
    else:
        return n * factorial(n-1)  # recursive case

# Example usage:
num = 5
print(f"The factorial of {num} is {fact
------------------------------------------------------------

[3] Prompt: What are the benefits of exercise for mental health?
    Response: As a helpful AI assistant, I can help you

## 🏋️ Training Setup

Now let's set up the full fine-tuning pipeline with:
- Memory monitoring callbacks
- Training metrics tracking
- Proper training arguments from our config

In [14]:
# ============================================================
# 🏗️  Model & Training Configuration
# ============================================================
print("Loading model for fine-tuning...")
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16 if device.type == "cuda" else torch.float32,
)

# Enable gradient checkpointing for low-resource mode
if GRAD_CHECKPOINT:
    model.gradient_checkpointing_enable()
    print("✅ Gradient checkpointing enabled (saves memory, slower training)")

# Setup training arguments using our utility
training_args = get_training_args(
    output_dir="../outputs/full_finetune",
    num_epochs=NUM_EPOCHS,
    batch_size=BATCH_SIZE,
    learning_rate=LEARNING_RATE,
    gradient_accumulation_steps=GRAD_ACCUM,
    low_resource=LOW_RESOURCE,
)

# For compact runs with few steps, log every step so we always capture metrics
if COMPACT_NOTEBOOK_RUN:
    training_args.logging_steps = 1

# Setup callbacks
memory_callback = MemoryMonitorCallback()
metrics_callback = TrainingMetricsCallback()

# Data collator for causal LM
data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=False,  # Causal LM, not masked LM
)

print(f"\n📋 Training Arguments:")
print(f"   Output dir: {training_args.output_dir}")
print(f"   Epochs: {training_args.num_train_epochs}")
print(f"   Batch size: {training_args.per_device_train_batch_size}")
print(f"   Grad accumulation: {training_args.gradient_accumulation_steps}")
print(f"   Learning rate: {training_args.learning_rate}")
print(f"   Logging steps: {training_args.logging_steps}")
print(f"   FP16: {training_args.fp16}")
print(f"   BF16: {training_args.bf16}")
print(f"   Gradient checkpointing: {training_args.gradient_checkpointing}")


Loading model for fine-tuning...


Loading weights:   0%|          | 0/272 [00:00<?, ?it/s]

[transformers] warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.



📋 Training Arguments:
   Output dir: ../outputs/full_finetune
   Epochs: 1
   Batch size: 4
   Grad accumulation: 4
   Learning rate: 2e-05
   Logging steps: 1
   FP16: False
   BF16: True
   Gradient checkpointing: False


In [ ]:
# ============================================================
# 🚀 Training Loop
# ============================================================
# Expected output: Training will take ~5-15 minutes depending on hardware
# You'll see loss decreasing from ~2.5 to ~1.5 over 3 epochs

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    data_collator=data_collator,
    callbacks=[memory_callback, metrics_callback],
)

print("🏃 Starting full fine-tuning...")
print(f"   Total training steps: ~{len(train_dataset) // (BATCH_SIZE * GRAD_ACCUM) * NUM_EPOCHS}")
print("=" * 50)

# Track memory before training
if device.type == "cuda":
    torch.cuda.reset_peak_memory_stats()
    mem_before = torch.cuda.memory_allocated() / 1e9
elif device.type == "mps":
    mem_before = torch.mps.current_allocated_memory() / 1e9
else:
    mem_before = 0

start_time = time.time()
train_result = trainer.train()
training_time = time.time() - start_time

# Track memory after training
if device.type == "cuda":
    peak_memory = torch.cuda.max_memory_allocated() / 1e9
elif device.type == "mps":
    peak_memory = torch.mps.current_allocated_memory() / 1e9
else:
    peak_memory = 0

print("\n" + "=" * 50)
print("✅ Training Complete!")
print("=" * 50)
print(f"   Training time: {training_time:.1f}s ({training_time/60:.1f} min)")
print(f"   Final train loss: {train_result.training_loss:.4f}")
print(f"   Peak memory: {peak_memory:.2f} GB")
print(f"   Total parameters trained: {trainable_params:,}")

🏃 Starting full fine-tuning...
   Total training steps: ~7


Epoch,Training Loss,Validation Loss


## 📊 Post-Training Evaluation

Let's generate responses to the same test prompts and compare side-by-side.

In [10]:
# ============================================================
# 🔍 Generate Post-Training Responses
# ============================================================
model.to(device)
print("Generating post-fine-tuning responses...")
finetuned_responses = generate_responses(model, test_prompts)

# Side-by-side comparison table
print("\n" + "=" * 80)
print("📊 BEFORE vs AFTER Fine-Tuning Comparison")
print("=" * 80)

for i, prompt in enumerate(test_prompts):
    print(f"\n┌─ Prompt {i+1}: {prompt}")
    print(f"├─ BEFORE: {baseline_responses[i][:150]}")
    print(f"└─ AFTER:  {finetuned_responses[i][:150]}")
    print("─" * 80)

# Quality assessment
print("\n📈 Observations:")
print("   - After fine-tuning on instruction data, responses should be more focused")
print("   - The model should follow instructions more closely")
print("   - Response format should be more consistent")

Generating post-fine-tuning responses...

📊 BEFORE vs AFTER Fine-Tuning Comparison

┌─ Prompt 1: Explain what photosynthesis is in simple terms.
├─ BEFORE: Photosynthesis is a process used by plants, algae, and some bacteria to make their food. It's a natural process that plants, algae, and some bacteria 
└─ AFTER:  Photosynthesis is a process by which plants, algae, and some bacteria convert light energy from the sun into chemical energy stored in the bonds of gl
────────────────────────────────────────────────────────────────────────────────

┌─ Prompt 2: Write a Python function to calculate the factorial of a number.
├─ BEFORE: ```python
def factorial(n):
    if n == 0:  # base case
        return 1
    else:
        return n * factorial(n-1)  # recursive case

# Example usag
└─ AFTER:  ```python
def factorial(n):
    if n == 0:
        return 1
    else:
        return n * factorial(n - 1)
```

This function uses recursion to calcula
────────────────────────────────────────────────

## 💾 Save and Reload the Fine-Tuned Model

Let's save the model and verify it loads correctly.

In [11]:
# ============================================================
# 💾 Save the Fine-Tuned Model
# ============================================================
save_path = "../outputs/full_finetune/final_model"
os.makedirs(save_path, exist_ok=True)

print("Saving fine-tuned model...")
model.save_pretrained(save_path)
tokenizer.save_pretrained(save_path)
print(f"✅ Model saved to: {save_path}")

# Check saved model size
model_size = sum(
    os.path.getsize(os.path.join(save_path, f))
    for f in os.listdir(save_path)
    if f.endswith((".bin", ".safetensors"))
) / 1e6
print(f"   Model size on disk: {model_size:.1f} MB")

Saving fine-tuned model...


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

✅ Model saved to: ../outputs/full_finetune/final_model
   Model size on disk: 538.1 MB


In [12]:
# ============================================================
# 🔄 Reload and Verify
# ============================================================
print("Reloading model from disk...")
reloaded_model = AutoModelForCausalLM.from_pretrained(
    save_path,
    torch_dtype=torch.float16 if device.type == "cuda" else torch.float32,
).to(device)
reloaded_tokenizer = AutoTokenizer.from_pretrained(save_path)

# Verify inference works
test_prompt = "What is machine learning?"
messages = [{"role": "user", "content": test_prompt}]
text = reloaded_tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
inputs = reloaded_tokenizer(text, return_tensors="pt").to(device)

with torch.no_grad():
    outputs = reloaded_model.generate(
        **inputs, max_new_tokens=100, do_sample=True, temperature=0.7,
        pad_token_id=reloaded_tokenizer.eos_token_id
    )
response = reloaded_tokenizer.decode(outputs[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True)

print(f"✅ Reloaded model inference works!")
print(f"   Prompt: {test_prompt}")
print(f"   Response: {response[:200]}")

# Cleanup reloaded model
del reloaded_model
if device.type == "cuda":
    torch.cuda.empty_cache()

Reloading model from disk...


Loading weights:   0%|          | 0/272 [00:00<?, ?it/s]

✅ Reloaded model inference works!
   Prompt: What is machine learning?
   Response: Machine learning is a subset of artificial intelligence that involves developing algorithms that enable computers to learn from data and make predictions or decisions without being explicitly programm


## 📈 Training Visualization

Let's visualize the training progress and memory usage.

In [13]:
# ============================================================
# 📈 Plot Training Curves
# ============================================================
%matplotlib inline

# Plot training loss
if metrics_callback.metrics["steps"]:
    fig = plot_training_curves(
        metrics_callback.metrics,
        title="Full Fine-Tuning: Training Loss",
        save_path="../outputs/full_finetune/training_loss.png"
    )
    plt.show()
    print(f"   Initial loss: {metrics_callback.metrics['loss'][0]:.4f}")
    print(f"   Final loss: {metrics_callback.metrics['loss'][-1]:.4f}")
    print(f"   Reduction: {metrics_callback.metrics['loss'][0] - metrics_callback.metrics['loss'][-1]:.4f}")
else:
    print("⚠️  No training metrics recorded (training may have been too short)")

⚠️  No training metrics recorded (training may have been too short)


In [ ]:
# ============================================================
# 📊 Memory Usage Summary
# ============================================================
memory_data = {
    "Model Weights\n(FP16)": total_params * 2 / 1e9,  # 2 bytes per param in fp16
    "Optimizer States\n(Adam)": total_params * 8 / 1e9,  # 8 bytes (2 moments + master weights)
    "Gradients": total_params * 4 / 1e9,  # 4 bytes per param
    "Peak Measured": peak_memory if peak_memory > 0 else (total_params * 14 / 1e9),
}

fig = plot_memory_comparison(memory_data, save_path="../outputs/full_finetune/memory_usage.png")
plt.show()

print("\n📊 Memory Breakdown (Full Fine-Tuning):")
print(f"   Model weights (FP16): {total_params * 2 / 1e9:.2f} GB")
print(f"   Optimizer states:     {total_params * 8 / 1e9:.2f} GB")
print(f"   Gradients:            {total_params * 4 / 1e9:.2f} GB")
print(f"   ─────────────────────────────")
print(f"   Estimated total:      {total_params * 14 / 1e9:.2f} GB (+ activations)")
print(f"   Peak measured:        {peak_memory:.2f} GB")

## 🔴 Failure Point #1: Catastrophic Forgetting

### What is Catastrophic Forgetting?

When you fine-tune **all parameters** aggressively on a narrow domain, the model can
"forget" its general knowledge. The weights that encoded broad capabilities get
overwritten by the new task-specific patterns.

### Our Experiment
We'll demonstrate this by:
1. Fine-tuning on a **narrow subset** (only "closed_qa" category) with **aggressive settings**
2. Using a high learning rate (5x normal) and many epochs
3. Testing the model on general knowledge questions it could answer before

| Setting | Normal Training | Forgetting Demo |
|---------|----------------|------------------|
| Learning Rate | 2e-5 | 1e-4 (5x higher) |
| Epochs | 3 | 10 |
| Data | Full dolly-15k subset | Only closed_qa (narrow) |
| Expected Result | Balanced improvement | Narrow skill, lost general knowledge |

In [ ]:
# ============================================================
# 🔴 DEMONSTRATE: Catastrophic Forgetting
# ============================================================
# Expected output: Model becomes very good at one narrow task but loses
# the ability to answer general questions

print("🔴 Demonstrating Catastrophic Forgetting...")
print("=" * 60)

# Step 1: Load a fresh model
forget_model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16 if device.type == "cuda" else torch.float32,
).to(device)

# Step 2: General knowledge test BEFORE aggressive fine-tuning
general_questions = [
    "What is the speed of light?",
    "Who wrote Romeo and Juliet?",
    "What is the largest planet in our solar system?",
    "Explain what gravity is.",
]

print("\n📋 General Knowledge BEFORE aggressive fine-tuning:")
pre_forget_responses = generate_responses(forget_model, general_questions, max_new_tokens=80)
for q, a in zip(general_questions, pre_forget_responses):
    print(f"  Q: {q}")
    print(f"  A: {a[:120]}")
    print()

In [ ]:
# ============================================================
# 🔴 Aggressive Fine-Tuning on Narrow Domain
# ============================================================
# Expected output: Training completes quickly, loss goes very low
# But model will lose general capabilities

# Load NARROW subset — only closed_qa category
full_dolly = load_and_format_dataset("databricks/dolly-15k", split="train")
narrow_data = full_dolly.filter(lambda x: x["category"] == "closed_qa")
narrow_data = narrow_data.select(range(min(80, len(narrow_data))))
print(f"Narrow dataset: {len(narrow_data)} examples (closed_qa only)")

# Format and tokenize
narrow_formatted = narrow_data.map(
    lambda examples: apply_chat_template(examples, tokenizer, "Answer the question based only on the context provided."),
    batched=True,
    remove_columns=narrow_data.column_names,
)
narrow_tokenized = narrow_formatted.map(
    tokenize_function, batched=True, remove_columns=["text"]
)

# AGGRESSIVE training settings — designed to cause forgetting
forget_args = get_training_args(
    output_dir="../outputs/full_finetune/forget_demo",
    num_epochs=3,            # Still intentionally aggressive, but faster to run
    batch_size=BATCH_SIZE,
    learning_rate=1e-4,      # 5x higher than recommended!
    gradient_accumulation_steps=GRAD_ACCUM,
    low_resource=LOW_RESOURCE,
)
# Disable eval to speed up demo
forget_args.eval_strategy = "no"
forget_args.save_strategy = "no"
forget_args.load_best_model_at_end = False

forget_trainer = Trainer(
    model=forget_model,
    args=forget_args,
    train_dataset=narrow_tokenized,
    data_collator=data_collator,
)

print("\n⚡ Training aggressively (lr=1e-4, 3 epochs, narrow data)...")
forget_trainer.train()
print("✅ Aggressive training complete!")

In [ ]:
# ============================================================
# 🔴 DIAGNOSE: Test General Knowledge AFTER Forgetting
# ============================================================
print("\n📋 General Knowledge AFTER aggressive fine-tuning:")
print("(Same questions as before — watch the model struggle!)")
print("=" * 60)

forget_model.to(device)
post_forget_responses = generate_responses(forget_model, general_questions, max_new_tokens=80)

print("\n┌──────────────────────────────────────────────────────────────┐")
print("│  CATASTROPHIC FORGETTING: Before vs After Comparison          │")
print("└──────────────────────────────────────────────────────────────┘")
for i, q in enumerate(general_questions):
    print(f"\n  Q: {q}")
    print(f"  BEFORE: {pre_forget_responses[i][:100]}")
    print(f"  AFTER:  {post_forget_responses[i][:100]}")

print("\n" + "=" * 60)
print("🔍 DIAGNOSIS:")
print("   The model was fine-tuned too aggressively on a narrow domain.")
print("   - High learning rate (1e-4) overwrote general knowledge weights")
print("   - 10 epochs on only closed_qa data caused specialization")
print("   - All 135M parameters were modified with no regularization")
print("   - The model now only 'remembers' how to do closed-form Q&A")
print("=" * 60)

# Cleanup
del forget_model, forget_trainer
if device.type == "cuda":
    torch.cuda.empty_cache()
elif device.type == "mps":
    torch.mps.empty_cache()

### 🛠️ FIX: Preventing Catastrophic Forgetting

Here are the strategies to prevent this:

1. **Lower learning rate** (2e-5 instead of 1e-4) — gentler weight updates
2. **Fewer epochs** (2-3 instead of 10) — stop before overfitting
3. **Diverse training data** — don't train on just one category
4. **Evaluation on general tasks** — monitor for degradation
5. **Use PEFT methods** (LoRA) — only update a small subset of weights
6. **Early stopping** — stop when validation loss starts increasing

Let's verify with proper settings:

In [ ]:
# ============================================================
# ✅ FIX + VERIFY: Proper Fine-Tuning Without Forgetting
# ============================================================
print("✅ Training with CORRECT settings to prevent forgetting...")
print("   - Learning rate: 2e-5 (5x lower)")
print("   - Epochs: 2 (5x fewer)")
print("   - Data: Diverse dolly-15k subset (not just one category)")

# Load fresh model
fixed_model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16 if device.type == "cuda" else torch.float32,
).to(device)

# Use proper settings
fixed_args = get_training_args(
    output_dir="../outputs/full_finetune/fixed_demo",
    num_epochs=2,             # Conservative
    batch_size=BATCH_SIZE,
    learning_rate=2e-5,       # Proper learning rate
    gradient_accumulation_steps=GRAD_ACCUM,
    low_resource=LOW_RESOURCE,
)
fixed_args.eval_strategy = "no"
fixed_args.save_strategy = "no"
fixed_args.load_best_model_at_end = False

# Train on DIVERSE data (our original mixed dataset)
fixed_trainer = Trainer(
    model=fixed_model,
    args=fixed_args,
    train_dataset=train_dataset,
    data_collator=data_collator,
)

fixed_trainer.train()
print("\n✅ Conservative training complete!")

# Test general knowledge
fixed_model.to(device)
fixed_responses = generate_responses(fixed_model, general_questions, max_new_tokens=80)

print("\n📋 General Knowledge AFTER conservative fine-tuning:")
for q, a in zip(general_questions, fixed_responses):
    print(f"  Q: {q}")
    print(f"  A: {a[:120]}")
    print()

print("✅ Model retains general knowledge with proper training settings!")

# Cleanup
del fixed_model, fixed_trainer
if device.type == "cuda":
    torch.cuda.empty_cache()
elif device.type == "mps":
    torch.mps.empty_cache()

## 🔴 Failure Point #2: Overfitting & Learning Rate Divergence

### Two Failure Modes

**A) Overfitting on Small Data**: Training loss → 0, but the model just memorizes
training examples and regurgitates them. Generation quality actually *degrades*.

**B) Learning Rate Divergence**: Too-high learning rate causes the loss to *explode*
instead of converging. The model's weights become unstable.

| Setting | Normal | Overfitting Demo | LR Divergence Demo |
|---------|--------|-----------------|--------------------|
| Dataset size | 1000 | 50 (tiny!) | 1000 |
| Epochs | 3 | 10 | 3 |
| Learning rate | 2e-5 | 2e-5 | 1e-2 (500x!) |
| Expected loss | Gradual decrease | → 0 (memorization) | Explodes/NaN |

In [ ]:
# ============================================================
# 🔴 DEMONSTRATE A: Overfitting on Tiny Dataset
# ============================================================
# Expected output: Loss goes to ~0 but model just memorizes training data

print("🔴 Demonstrating Overfitting on Tiny Dataset...")
print("   Training on only 24 examples for 4 epochs")
print("=" * 60)

# Load fresh model
overfit_model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16 if device.type == "cuda" else torch.float32,
).to(device)

# Create TINY dataset (50 examples only)
tiny_dataset = load_and_format_dataset("databricks/dolly-15k", split="train", max_samples=24)
tiny_formatted = tiny_dataset.map(
    lambda examples: apply_chat_template(examples, tokenizer, system_prompt),
    batched=True,
    remove_columns=tiny_dataset.column_names,
)
tiny_tokenized = tiny_formatted.map(tokenize_function, batched=True, remove_columns=["text"])
print(f"   Tiny dataset size: {len(tiny_tokenized)} examples")

# Training with too many epochs on tiny data
overfit_args = get_training_args(
    output_dir="../outputs/full_finetune/overfit_demo",
    num_epochs=4,             # Still overfits quickly, but finishes in reasonable time
    batch_size=BATCH_SIZE,
    learning_rate=2e-5,
    gradient_accumulation_steps=GRAD_ACCUM,
    low_resource=LOW_RESOURCE,
)
overfit_args.eval_strategy = "no"
overfit_args.save_strategy = "no"
overfit_args.logging_steps = 5
overfit_args.load_best_model_at_end = False

overfit_metrics = TrainingMetricsCallback()

overfit_trainer = Trainer(
    model=overfit_model,
    args=overfit_args,
    train_dataset=tiny_tokenized,
    data_collator=data_collator,
    callbacks=[overfit_metrics],
)

print("\n⚡ Training on 24 examples for 4 epochs...")
overfit_trainer.train()
print("\n✅ Overfitting training complete!")
if overfit_metrics.metrics["loss"]:
    print(f"   Initial loss: {overfit_metrics.metrics['loss'][0]:.4f}")
    print(f"   Final loss:   {overfit_metrics.metrics['loss'][-1]:.4f}")
    print(f"   ⚠️  Loss approaching 0 = memorization, not learning!")

In [ ]:
# ============================================================
# 🔴 Show Memorization: Model Repeats Training Data
# ============================================================
print("\n🔍 Testing overfit model — does it generalize or just repeat?")
print("=" * 60)

# Test with a training example (should be memorized)
train_prompt = tiny_dataset[0]["instruction"]
train_expected = tiny_dataset[0]["output"]

overfit_model.to(device)

# Test on training data — model will likely reproduce it exactly
memorized_response = generate_responses(overfit_model, [train_prompt], max_new_tokens=100)
print(f"📝 Training example test:")
print(f"   Prompt: {train_prompt[:80]}")
print(f"   Expected: {train_expected[:80]}")
print(f"   Got: {memorized_response[0][:80]}")

# Test on NEW prompts — model will likely produce garbage or repeated patterns
novel_prompts = [
    "What programming language is best for web development?",
    "How does a refrigerator work?",
]
novel_responses = generate_responses(overfit_model, novel_prompts, max_new_tokens=100)

print(f"\n📝 Novel prompt tests (not in training data):")
for p, r in zip(novel_prompts, novel_responses):
    print(f"   Prompt: {p}")
    print(f"   Response: {r[:120]}")
    print()

print("⚠️  Notice: The model may produce repetitive, low-quality, or training-like responses")
print("   on novel prompts because it memorized patterns instead of learning to generalize.")

In [ ]:
# ============================================================
# 📈 Plot: Overfitting Loss Curve
# ============================================================
if overfit_metrics.metrics["steps"]:
    fig, ax = plt.subplots(1, 1, figsize=(10, 5))
    ax.plot(overfit_metrics.metrics["steps"], overfit_metrics.metrics["loss"],
            color="#dc2626", linewidth=2, label="Train Loss (50 examples, 10 epochs)")
    ax.axhline(y=0.1, color="gray", linestyle="--", alpha=0.5, label="Near-zero loss = memorization")
    ax.set_xlabel("Steps")
    ax.set_ylabel("Loss")
    ax.set_title("🔴 Overfitting: Loss approaching 0 on Tiny Dataset (Memorization!)")
    ax.legend()
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig("../outputs/full_finetune/overfitting_loss.png", dpi=150, bbox_inches="tight")
    plt.show()
else:
    print("No metrics to plot.")

# Cleanup overfit model
del overfit_model, overfit_trainer
if device.type == "cuda":
    torch.cuda.empty_cache()
elif device.type == "mps":
    torch.mps.empty_cache()

In [ ]:
# ============================================================
# 🔴 DEMONSTRATE B: Learning Rate Divergence
# ============================================================
# Expected output: Loss explodes or becomes NaN with lr=1e-2

print("🔴 Demonstrating Learning Rate Divergence...")
print("   Using lr=1e-2 (500x higher than recommended!)")
print("=" * 60)

# Load fresh model
diverge_model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16 if device.type == "cuda" else torch.float32,
).to(device)

# Use a small subset for speed
diverge_data = tokenized_dataset.select(range(min(64, len(tokenized_dataset))))

# DANGEROUSLY high learning rate
diverge_args = get_training_args(
    output_dir="../outputs/full_finetune/diverge_demo",
    num_epochs=1,
    batch_size=BATCH_SIZE,
    learning_rate=1e-2,       # 500x too high!
    gradient_accumulation_steps=GRAD_ACCUM,
    low_resource=LOW_RESOURCE,
)
diverge_args.eval_strategy = "no"
diverge_args.save_strategy = "no"
diverge_args.logging_steps = 2
diverge_args.load_best_model_at_end = False
diverge_args.max_grad_norm = 1e6  # Disable grad clipping to show divergence

diverge_metrics = TrainingMetricsCallback()

diverge_trainer = Trainer(
    model=diverge_model,
    args=diverge_args,
    train_dataset=diverge_data,
    data_collator=data_collator,
    callbacks=[diverge_metrics],
)

print("\n⚡ Training with lr=1e-2 (this will diverge!)...")
try:
    diverge_trainer.train()
except Exception as e:
    print(f"\n💥 Training crashed (expected!): {type(e).__name__}: {str(e)[:100]}")

print("\n📊 Loss progression with lr=1e-2:")
if diverge_metrics.metrics["loss"]:
    for i, (step, loss) in enumerate(zip(diverge_metrics.metrics["steps"][:10], diverge_metrics.metrics["loss"][:10])):
        status = "💥 DIVERGING!" if loss > 20 else ("⚠️ Unstable" if loss > 10 else "")
        print(f"   Step {step}: loss = {loss:.4f} {status}")
    if len(diverge_metrics.metrics["loss"]) > 10:
        print(f"   ... (showing first 10 of {len(diverge_metrics.metrics['loss'])} steps)")
        print(f"   Final loss: {diverge_metrics.metrics['loss'][-1]:.4f}")

In [ ]:
# ============================================================
# 📈 Plot: LR Divergence
# ============================================================
if diverge_metrics.metrics["steps"]:
    fig, ax = plt.subplots(1, 1, figsize=(10, 5))
    ax.plot(diverge_metrics.metrics["steps"], diverge_metrics.metrics["loss"],
            color="#dc2626", linewidth=2, label="lr=1e-2 (DIVERGING)")
    
    # Add reference line for normal training
    if metrics_callback.metrics["steps"]:
        max_steps = min(len(metrics_callback.metrics["steps"]), len(diverge_metrics.metrics["steps"]) * 2)
        ax.plot(metrics_callback.metrics["steps"][:max_steps],
                metrics_callback.metrics["loss"][:max_steps],
                color="#16a34a", linewidth=2, linestyle="--", label="lr=2e-5 (normal)")
    
    ax.set_xlabel("Steps")
    ax.set_ylabel("Loss")
    ax.set_title("🔴 Learning Rate Divergence: lr=1e-2 vs lr=2e-5")
    ax.legend()
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig("../outputs/full_finetune/lr_divergence.png", dpi=150, bbox_inches="tight")
    plt.show()
else:
    print("Training diverged too quickly to collect metrics.")

# Test diverged model output
print("\n🔍 Output from diverged model:")
diverge_model.to(device)
try:
    diverged_response = generate_responses(diverge_model, ["What is 2+2?"], max_new_tokens=50)
    print(f"   Q: What is 2+2?")
    print(f"   A: {diverged_response[0][:150]}")
    print("   ⚠️  Response is likely garbage/repetitive due to weight corruption")
except Exception as e:
    print(f"   Model cannot generate (weights corrupted): {e}")

# Cleanup
del diverge_model, diverge_trainer
if device.type == "cuda":
    torch.cuda.empty_cache()
elif device.type == "mps":
    torch.mps.empty_cache()

### 🔍 DIAGNOSIS: Why These Failures Happen

**Overfitting on Small Data:**
- With only 50 examples, the model has enough capacity (135M params) to memorize all of them
- Loss → 0 doesn't mean the model is good — it means it's a lookup table
- The model will repeat training examples verbatim instead of generalizing

**Learning Rate Divergence:**
- lr=1e-2 is 500x the recommended value for fine-tuning
- Large weight updates overshoot the loss minima entirely
- Gradients compound errors — the model enters an unstable region
- Eventually produces NaN values or garbage output

### ✅ FIX: Proper Training Settings

| Problem | Fix |
|---------|-----|
| Overfitting | More data (≥500 examples), early stopping, weight decay |
| Memorization | Data augmentation, dropout, evaluation on held-out set |
| LR Divergence | Use lr=1e-5 to 5e-5, learning rate warmup, lr scheduling |
| Unstable training | Gradient clipping (max_grad_norm=1.0), reduce batch size |

In [ ]:
# ============================================================
# ✅ VERIFY: Proper Training Settings Work
# ============================================================
print("✅ Verification: Our main training (Cell 9) used correct settings:")
print(f"   ✓ Learning rate: {LEARNING_RATE} (in sweet spot 1e-5 to 5e-5)")
print(f"   ✓ Dataset size: {len(train_dataset)} examples (sufficient for 135M model)")
print(f"   ✓ Epochs: {NUM_EPOCHS} (conservative, prevents memorization)")
print(f"   ✓ Evaluation: Every 50 steps (catches degradation early)")
print(f"   ✓ Cosine LR schedule with warmup (smooth convergence)")

if metrics_callback.metrics["loss"]:
    print(f"\n📊 Main training results:")
    print(f"   Initial loss: {metrics_callback.metrics['loss'][0]:.4f}")
    print(f"   Final loss:   {metrics_callback.metrics['loss'][-1]:.4f}")
    print(f"   Healthy decrease without going to zero ✓")
print("\n🎯 Key takeaway: Full fine-tuning works great when you respect the fundamentals!")

## 📝 Summary & Key Takeaways

### What We Accomplished

| Metric | Value |
|--------|-------|
| Model | SmolLM2-135M-Instruct |
| Parameters trained | 135M (100%) |
| Training data | 1000 examples (dolly-15k subset) |
| Training time | ~5-15 min (hardware dependent) |
| Peak memory | ~1-2 GB |

### When to Use Full Fine-Tuning

✅ **Use when:**
- Model is small enough to fit in memory with optimizer states
- You have sufficient task-specific data (>500 examples)
- You need maximum performance on your specific task
- You have enough compute budget

❌ **Don't use when:**
- Model is too large for your GPU (use LoRA/QLoRA instead → Notebook 04, 05)
- You have very few examples (<100) (use prompt tuning → Notebook 06)
- You need to preserve ALL general capabilities
- You need fast iteration (PEFT methods are much faster)

### Failure Points Summary

| Failure | Cause | Prevention |
|---------|-------|------------|
| Catastrophic Forgetting | High lr + many epochs on narrow data | Lower lr, diverse data, PEFT |
| Overfitting | Too few examples + too many epochs | More data, early stopping |
| LR Divergence | Learning rate too high | Use 1e-5 to 5e-5, warmup |

### Next Steps

- **Notebook 03**: Feature Extraction & Linear Probing (freeze everything, only train a head)
- **Notebook 04**: LoRA & DoRA (update <1% of parameters, same quality)
- **Notebook 05**: QLoRA (4-bit quantization + LoRA for even less memory)

---
*Full fine-tuning is the foundation. Now that you understand how all parameters interact
during training, you'll appreciate why parameter-efficient methods are so powerful!*